##### Features and targets

Here, we prepare the model features for **every trading day**. A row dated *t* uses only data dated *t* or earlier.

**Target.** Notice that the one-day target is already in the panel: `fwd_spy_xs` = SPY minus BIL total return over
close(t+1) to close(t+2).

Output: `data/processed/features.parquet` (with a `.csv` version) and `data/processed/feature_sets.json`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))

from src import config
from src.data.build import load_panel
from src.features import events

panel = load_panel()
s = panel["in_sample"] # samples without the warmup period
print(panel.shape, panel.index.min().date(), "->", panel.index.max().date(), f"({s.sum()} trading days in sample)")

(3530, 34) 2012-01-03 -> 2026-01-15 (3018 trading days in sample)


In [2]:
cols = ["spy_adj_close", "spy_tr_cc", "vix_close", "vix3m_close", "vix_ratio", "fwd_spy_xs"]
display(panel.loc[s, cols].describe().T.round(4))
print("NaN in sample:", int(panel.loc[s, cols].isna().sum().sum()))
print("fwd_spy_xs[t] == spy_xs_cc[t+2]:",
      np.allclose(panel["fwd_spy_xs"], panel["spy_xs_cc"].shift(-2), equal_nan=True))

,count,mean,std,min,25%,50%,75%,max
spy_adj_close,3018.0,322.8707,143.6480,141.7395,195.5039,272.9063,416.0080,690.3800
spy_tr_cc,3018.0,0.0006,0.0109,-0.1094,-0.0036,0.0007,0.0058,0.1050
vix_close,3018.0,17.9843,6.9631,9.1400,13.3700,16.1500,20.7100,82.6900
vix3m_close,3018.0,19.9033,6.1617,11.8500,15.6000,18.2750,22.3725,72.9800
vix_ratio,3018.0,0.8911,0.0778,0.7359,0.8368,0.8798,0.9351,1.3437
fwd_spy_xs,3018.0,0.0005,0.0109,-0.1095,-0.0037,0.0007,0.0057,0.1050


NaN in sample: 0
fwd_spy_xs[t] == spy_xs_cc[t+2]: True


#### 2. Feature definitions

Here, we propose candidate features in the following categorised order. Three of them are dropped after the
collinearity check in section 3; they are kept in the code so the check can be reproduced.

- **Decline state**: how far and how fast SPY has fallen.
  - `dec_dd`: t_close / 20-day high − 1.
  - `dec_mom`: 5-day return / (20-day vol × √5). 
    - This represents a z-score of the 5-day return
  - `dec_vol`: std of daily SPY total returns over the last 20 days.
  - `dec_cond`: the decline trigger, 1 if both conditions hold, else 0.
    - `dec_dd` ≤ −2%
    - `dec_mom` ≤ −1
- **Trend**: whether the longer-run uptrend is intact.
  - `ma200_gap`: close / 200-day moving average − 1.
- **VIX level**: how much fear is priced in, and in which direction it is moving.
  - `vix_log`: log VIX.
  - `vix_z`: (VIX − 252-day mean) / 252-day std, i.e. VIX relative to its own past year.
  - `vix_chg5`: log change of VIX over 5 days.
- **Term structure**: whether the market sees the stress as short-lived or lasting.
  - `ts_ratio`: VIX / VIX3M.
    - Above 1 means backwardation.
  - `ts_backw`: 1 if `ts_ratio` > 1, else 0.
  - `ts_chg5`: 5-day change of `ts_ratio`. *(dropped in the pruning step below)*
- **Implied vs realised volatility**: whether options price more fear than the market has shown.
  - `vrp`: VIX/100 − annualised 20-day realised vol.
- **Interactions**: does the volatility signal behave differently during declines?
  - With the trigger flag:
    - `x_cond_ts`: `dec_cond` × (`ts_ratio` − 1)
    - `x_cond_vrp`: `dec_cond` × `vrp` *(dropped)*
    - `x_cond_vixchg`: `dec_cond` × `vix_chg5` *(dropped)*
  - With continuous decline measures:
    - `x_dd_ts`: `dec_dd` × (`ts_ratio` − 1)
    - `x_mom_backw`: `dec_mom` × `ts_backw`

Notice that we transform the term-structure ratio a bit. Since the raw reading are around 0.9, the interaction term `dec_cond × ts_ratio` would be almost the same as `dec_cond`.

In [3]:
# The functions live in src/features/feature_build.py so scripts and tests can import them.
# Shown here for reading; edit the module, not this cell.
import inspect
from src.features import feature_build as fb

for fn in (fb.decline_features, fb.vix_features, fb.interactions, fb.build_candidates):
    print(inspect.getsource(fn))

feat = fb.build_candidates(panel)          # all 17 candidates, before pruning
feat.loc[s].describe().T.round(4)

def decline_features(panel: pd.DataFrame) -> pd.DataFrame:
    f = events.decline_signals(panel)                          # dec_dd, dec_mom, dec_vol
    f["dec_cond"] = ((f["dec_dd"] <= -events.S) & (f["dec_mom"] <= -events.P)).astype(float)
    a = panel["spy_adj_close"]
    f["ma200_gap"] = a / a.rolling(200).mean() - 1
    return f

def vix_features(panel: pd.DataFrame) -> pd.DataFrame:
    vix, ratio = panel["vix_close"], panel["vix_ratio"]
    return pd.DataFrame({
        "vix_log": np.log(vix),
        "vix_z": (vix - vix.rolling(252).mean()) / vix.rolling(252).std(),
        "vix_chg5": np.log(vix / vix.shift(5)),
        "ts_ratio": ratio,
        "ts_backw": (ratio > 1).astype(float),
        "ts_chg5": ratio - ratio.shift(5),
    })

def interactions(f: pd.DataFrame) -> pd.DataFrame:
    ts = f["ts_ratio"] - 1          # distance from backwardation; a fixed centre, so no leakage
    return pd.DataFrame({
        "x_cond_ts": f["dec_cond"] * ts,
        "x_dd_ts": f["dec_dd"]

,count,mean,std,min,25%,50%,75%,max
dec_dd,3018.0,-0.0181,0.0289,-0.2911,-0.0235,-0.0070,-0.0003,0.0000
dec_mom,3018.0,0.2090,0.9743,-3.7506,-0.3819,0.2752,0.8738,3.4825
dec_vol,3018.0,0.0092,0.0061,0.0020,0.0056,0.0076,0.0110,0.0587
dec_cond,3018.0,0.0958,0.2943,0.0000,0.0000,0.0000,0.0000,1.0000
ma200_gap,3018.0,0.0525,0.0606,-0.2566,0.0260,0.0633,0.0909,0.1786
vix_log,3018.0,2.8329,0.3207,2.2127,2.5930,2.7819,3.0306,4.4151
vix_z,3018.0,-0.0346,1.2741,-2.0558,-0.8760,-0.3101,0.4635,11.8278
vix_chg5,3018.0,0.0001,0.1636,-0.6287,-0.0903,-0.0121,0.0765,1.1407
ts_ratio,3018.0,0.8911,0.0778,0.7359,0.8368,0.8798,0.9351,1.3437
ts_backw,3018.0,0.0759,0.2648,0.0000,0.0000,0.0000,0.0000,1.0000


#### 3. Checks

**Coverage.** Every in-sample day must have every feature (the 2012–13 warm-up covers the 252-day VIX window).

**Leakage.** Rebuild the features on the panel cut at several dates and confirm the values up to the cut are
identical.

In [4]:
nan_in_sample = feat.loc[s].isna().sum()
assert nan_in_sample.sum() == 0, nan_in_sample[nan_in_sample > 0]
assert not any(c.startswith("fwd_") for c in feat.columns)

for frac in (0.25, 0.5, 0.75, 0.95):
    cut = panel.index[int(len(panel) * frac)]
    pd.testing.assert_frame_equal(feat.loc[:cut], fb.build_candidates(panel.loc[:cut]), check_exact=True)
print("coverage and truncation checks passed")

coverage and truncation checks passed


**Collinearity.** Pairs of features that move almost together. Normally, the Ridge estimator copes with these, but very those high pairs
make individual coefficients hard to read.

In [5]:
c = feat.loc[s].corr()
pairs = c.where(np.triu(np.ones(c.shape, dtype=bool), k=1)).stack()
pairs[pairs.abs() > 0.7].sort_values(key=abs, ascending=False).round(3).to_frame("corr")

corr
vix_chg5    ts_chg5        0.921
vix_z       ts_ratio       0.841
x_cond_vrp  x_cond_vixchg  0.837
dec_cond    x_cond_vrp     0.837
            x_cond_vixchg  0.811
dec_vol     vix_log        0.754
dec_mom     vix_chg5      -0.748
x_mom_backw x_cond_vixchg -0.734
dec_dd      vix_z         -0.726
            ts_ratio      -0.708
dec_vol     vrp           -0.702

**Pruning.** Based on the correlations above, we drop three of the 17 candidates, leaving 14 features.

- `ts_chg5` (corr 0.92 with `vix_chg5`)
  - When VIX jumps, the front end rises much more than VIX3M, so the 5-day change of the ratio moves almost
    one-for-one with the 5-day change of VIX. The two columns carry nearly the same information.
  - We keep `vix_chg5`. The term structure is still represented by `ts_ratio` and `ts_backw`.
- `x_cond_vrp` (corr 0.84 with `dec_cond`) and `x_cond_vixchg` (corr 0.81 with `dec_cond`)
  - On trigger days `vrp` and `vix_chg5` almost always have the same sign (VIX nearly always rises during a
    decline), so multiplying them by `dec_cond` returns little more than a rescaled `dec_cond`.
  - Centring at a fixed constant, which fixed `x_cond_ts`, does not help when the sign barely varies, and centring
    at a sample mean would leak. Dropping them also keeps the interactions focused on the term structure, which
    is the project's research question.

Pairs we keep despite high correlation, because each side answers a different question:

- `vix_z` vs `ts_ratio` (0.84): VIX level vs term-structure shape. Whether the shape adds anything beyond the level
  is exactly what the `base` / `vix` / `full` comparison tests, so both stay. Their individual ridge coefficients
  should not be over-read, since ridge splits the weight between them.
- `dec_vol` vs `vix_log` (0.75), `dec_mom` vs `vix_chg5` (−0.75), `dec_dd` vs `vix_z` (−0.73) and `ts_ratio`
  (−0.71), `dec_vol` vs `vrp` (−0.70): the ordinary link between falling prices and rising implied volatility.
  These pairs are what separate the price-only `base` set from the `vix` set.

In [6]:
print("dropped:", fb.DROPPED)          # set in feature_build.py
feat = feat.drop(columns=fb.DROPPED)

c = feat.loc[s].corr()
pairs = c.where(np.triu(np.ones(c.shape, dtype=bool), k=1)).stack()
print(f"{feat.shape[1]} features kept; largest remaining |corr|:")
pairs.abs().sort_values(ascending=False).head(3).round(3)

dropped: ['ts_chg5', 'x_cond_vrp', 'x_cond_vixchg']
14 features kept; largest remaining |corr|:


vix_z    ts_ratio    0.841
dec_vol  vix_log     0.754
dec_mom  vix_chg5    0.748
dtype: float64

#### 4. Feature sets

Then, we sort the proposed features into three sets. 

- `base`: price-based decline state and trend only
- `vix`: `base` plus VIX level, term structure and the implied-vs-realised gap
- `full`: `vix` plus the decline × volatility interactions

In the later performance evaluation, we could statitsically test whether incorporating the interaction terms
brings in extra predictive power for the mode.

In [7]:
FEATURE_SETS = fb.FEATURE_SETS             # set in feature_build.py
assert set(FEATURE_SETS["full"]) == set(feat.columns)
pd.testing.assert_frame_equal(feat[FEATURE_SETS["full"]], fb.build_features(panel))
{k: len(v) for k, v in FEATURE_SETS.items()}

{'base': 5, 'vix': 11, 'full': 14}

**Decline days.** How many in-sample days have the trigger on, and how often it coincides with backwardation.

In [8]:
print(pd.crosstab(feat.loc[s, "dec_cond"], feat.loc[s, "ts_backw"],
                  rownames=["dec_cond"], colnames=["ts_backw"], margins=True))

ts_backw   0.0  1.0   All
dec_cond                 
0.0       2614  115  2729
1.0        175  114   289
All       2789  229  3018


#### 5. Save

In [9]:
# Same as running: python -m src.features.feature_build
fb.save(fb.build_features(panel))

wrote 14 features x 3530 rows to data/processed


To using the features, please do the following

```python
import json, pandas as pd
from src import config
from src.data.build import load_panel

panel = load_panel()
feat = pd.read_parquet(config.PROCESSED_DIR / "features.parquet")
sets = json.loads((config.PROCESSED_DIR / "feature_sets.json").read_text())
X, y = feat[sets["full"]], panel["fwd_spy_xs"]
```

Note that `fwd_spy_xs` at *t* is realised at close(t+2), so a model refitted at date *T* may only use rows with
*t* ≤ *T* − 2.